# V13: Accuracy Push Experiments

Baseline: V10 SOTA — 30.67% Top-1

Experiments:
1. Ridge Alpha Cross-Validation
2. Confidence-Weighted Ridge
3. CSLS Retrieval (with go/no-go hub check)
4. Pure 768d Ablation
5. GloVe 840B Target Space

In [1]:
import json
import pickle
import re
import numpy as np
from pathlib import Path
from collections import Counter
from gensim.models import KeyedVectors
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if REPO_ROOT.name == "heiro_v13":
    REPO_ROOT = REPO_ROOT.parent

RESULTS = {
    "sanity_check": {},
    "baseline_v10": {"top1": 30.67, "top5": 37.69, "top10": 41.49},
}
RESULTS_PATH = REPO_ROOT / "heiro_v13/results/experiment_results.json"

def save_results():
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_PATH, "w") as f:
        json.dump(RESULTS, f, indent=2)
    print(f"Results saved to {RESULTS_PATH}")

print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: /Users/crashy/Development/heiroglyphy


In [2]:
print("Loading V7 FastText vectors (768d)...")
text_embeddings = KeyedVectors.load_word2vec_format(
    str(REPO_ROOT / "heiro_v7_FastTextVisual/models/fasttext_v7.vec"),
    binary=False,
)
print(f"  ✓ {len(text_embeddings)} words, {text_embeddings.vector_size}d")

print("\nLoading V9 visual embeddings (768d)...")
with open(REPO_ROOT / "heiro_v9_use_visuals_again/data/processed/visual_embeddings_768d.pkl", "rb") as f:
    visual_embeddings = pickle.load(f)
print(f"  ✓ {len(visual_embeddings)} Gardiner codes")

print("\nLoading V10 Gardiner mapping...")
with open(REPO_ROOT / "heiro_v10_refinement/data/gardiner_mapping.json", "r") as f:
    gardiner_mapping = json.load(f)

trans_to_gardiner = {}
for code, trans_str in gardiner_mapping.items():
    parts = re.split(r"[,;]", trans_str)
    for part in parts:
        clean = re.sub(r"\(.*?\)", "", part).strip()
        if clean:
            trans_to_gardiner.setdefault(clean, []).append(code)
print(f"  ✓ {len(trans_to_gardiner)} transliterations mapped")

print("\nLoading anchors...")
with open(REPO_ROOT / "heiro_v5_getdata/data/processed/english_anchors.json", "r") as f:
    anchors = json.load(f)
print(f"  ✓ {len(anchors)} anchor pairs")

print("\nLoading GloVe 6B (300d)...")
glove_6b = KeyedVectors.load_word2vec_format(
    str(REPO_ROOT / "heiro_v5_getdata/data/processed/glove.6B.300d.txt"),
    binary=False, no_header=True,
)
print(f"  ✓ {len(glove_6b)} English words, {glove_6b.vector_size}d")

Loading V7 FastText vectors (768d)...


  ✓ 80662 words, 768d

Loading V9 visual embeddings (768d)...
  ✓ 115 Gardiner codes

Loading V10 Gardiner mapping...
  ✓ 185 transliterations mapped

Loading anchors...
  ✓ 8541 anchor pairs

Loading GloVe 6B (300d)...


  ✓ 400000 English words, 300d


In [3]:
print("Creating 1536d fused embeddings...")
fused_embeddings = {}
visual_match_count = 0

for word in tqdm(text_embeddings.index_to_key, desc="Fusing"):
    text_vec = text_embeddings[word]
    visual_vec = None
    codes = trans_to_gardiner.get(word)
    if codes:
        vecs = [visual_embeddings[c] for c in codes if c in visual_embeddings]
        if vecs:
            visual_vec = np.mean(vecs, axis=0)
            visual_match_count += 1
    if visual_vec is None:
        visual_vec = np.zeros(768)
    fused_embeddings[word] = np.concatenate([text_vec, visual_vec])

print(f"✓ {len(fused_embeddings)} fused embeddings")
print(f"Visual match rate: {visual_match_count}/{len(fused_embeddings)} ({visual_match_count/len(fused_embeddings)*100:.2f}%)")

# Prepare anchor arrays
X_all, Y_all, conf_all, anchor_pairs = [], [], [], []
for a in anchors:
    egy, eng = a["hieroglyphic"], a["english"].lower()
    if egy in fused_embeddings and eng in glove_6b:
        X_all.append(fused_embeddings[egy])
        Y_all.append(glove_6b[eng])
        conf_all.append(a["confidence"])
        anchor_pairs.append((egy, eng))

X_all = np.array(X_all)
Y_all = np.array(Y_all)
conf_all = np.array(conf_all)
print(f"\nValid anchors: {len(X_all)} / {len(anchors)} ({len(X_all)/len(anchors)*100:.1f}%)")

# Train/test split — MUST match V10 exactly
X_train, X_test, Y_train, Y_test, conf_train, conf_test, pairs_train, pairs_test = train_test_split(
    X_all, Y_all, conf_all, anchor_pairs, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Creating 1536d fused embeddings...


Fusing:   0%|          | 0/80662 [00:00<?, ?it/s]

✓ 80662 fused embeddings
Visual match rate: 68/80662 (0.08%)

Valid anchors: 6700 / 8541 (78.4%)
Train: 5360, Test: 1340


In [4]:
def evaluate(Y_pred, Y_test, pairs_test, english_kv, topn=10):
    """Evaluate Top-1/5/10 accuracy using nearest-neighbor retrieval."""
    correct = {1: 0, 5: 0, 10: 0}
    predictions = []

    for i in range(len(Y_pred)):
        pred_vec = Y_pred[i]
        true_word = pairs_test[i][1]
        neighbors = english_kv.similar_by_vector(pred_vec, topn=topn)
        words = [w for w, _ in neighbors]
        predictions.append(words[0])

        if true_word == words[0]:
            correct[1] += 1
        if true_word in words[:5]:
            correct[5] += 1
        if true_word in words[:topn]:
            correct[10] += 1

    n = len(Y_pred)
    return {
        "top1": round(correct[1] / n * 100, 2),
        "top5": round(correct[5] / n * 100, 2),
        "top10": round(correct[10] / n * 100, 2),
        "predictions": predictions,
    }


def evaluate_csls(Y_pred, Y_test, pairs_test, english_kv, k=10, topn=10):
    """Evaluate using CSLS retrieval instead of plain NN.
    
    CSLS(x, y) = 2*cos(x,y) - r_T(x)
    where r_T(x) = mean similarity of x to its k nearest English neighbors.
    This penalizes 'hub' words that are near everything.
    """
    en_words = english_kv.index_to_key
    en_matrix = english_kv.vectors
    en_norms = np.linalg.norm(en_matrix, axis=1, keepdims=True)
    en_normed = en_matrix / np.maximum(en_norms, 1e-10)

    pred_norms = np.linalg.norm(Y_pred, axis=1, keepdims=True)
    pred_normed = Y_pred / np.maximum(pred_norms, 1e-10)

    correct = {1: 0, 5: 0, 10: 0}

    for i in tqdm(range(len(Y_pred)), desc="CSLS eval"):
        pred_vec = pred_normed[i]
        true_word = pairs_test[i][1]

        # Cosine similarities to all English words
        sims = en_normed @ pred_vec

        # r_T(x): mean of k nearest English neighbors
        top_k_sims = np.partition(sims, -k)[-k:]
        r_t = top_k_sims.mean()

        # CSLS score
        csls_scores = 2 * sims - r_t

        # Top-N by CSLS
        top_indices = np.argpartition(csls_scores, -topn)[-topn:]
        top_indices = top_indices[np.argsort(csls_scores[top_indices])[::-1]]
        words = [en_words[idx] for idx in top_indices]

        if true_word == words[0]:
            correct[1] += 1
        if true_word in words[:5]:
            correct[5] += 1
        if true_word in words[:topn]:
            correct[10] += 1

    n = len(Y_pred)
    return {
        "top1": round(correct[1] / n * 100, 2),
        "top5": round(correct[5] / n * 100, 2),
        "top10": round(correct[10] / n * 100, 2),
    }

## Sanity Check: Reproduce V10 Baseline (30.67%)

In [5]:
print("=" * 60)
print("SANITY CHECK: Reproducing V10 baseline")
print("=" * 60)

baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_train, Y_train)
Y_pred_baseline = baseline_model.predict(X_test)

baseline_results = evaluate(Y_pred_baseline, Y_test, pairs_test, glove_6b)
print(f"\nV10 Reproduction:")
print(f"  Top-1:  {baseline_results['top1']}%  (expected: 30.67%)")
print(f"  Top-5:  {baseline_results['top5']}%  (expected: 37.69%)")
print(f"  Top-10: {baseline_results['top10']}%  (expected: 41.49%)")

delta = abs(baseline_results["top1"] - 30.67)
if delta > 0.1:
    print(f"\n⚠️  SANITY CHECK FAILED — delta is {delta:.2f}%")
    print("    Data may have drifted. Investigate before proceeding.")
else:
    print(f"\n✓ SANITY CHECK PASSED (delta: {delta:.2f}%)")

RESULTS["sanity_check"] = {
    **baseline_results,
    "anchors_loaded": len(anchors),
    "valid_anchors": len(X_all),
    "train_size": len(X_train),
    "test_size": len(X_test),
    "passed": delta <= 0.1,
}
del RESULTS["sanity_check"]["predictions"]
save_results()

SANITY CHECK: Reproducing V10 baseline



V10 Reproduction:
  Top-1:  30.67%  (expected: 30.67%)
  Top-5:  37.69%  (expected: 37.69%)
  Top-10: 41.49%  (expected: 41.49%)

✓ SANITY CHECK PASSED (delta: 0.00%)
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 1: Ridge Alpha Cross-Validation

Hypothesis: alpha=1.0 is suboptimal for the ~86:1 param/sample ratio (460K params, ~5360 samples).

In [6]:
print("=" * 60)
print("EXPERIMENT 1: Ridge Alpha Cross-Validation")
print("=" * 60)

alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]

ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring="neg_mean_squared_error")
ridge_cv.fit(X_train, Y_train)

best_alpha = float(ridge_cv.alpha_)
print(f"\nBest alpha: {best_alpha}")
print(f"  (V10 used alpha=1.0)")

# Evaluate with best alpha
ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train, Y_train)
Y_pred_exp1 = ridge_best.predict(X_test)

exp1_results = evaluate(Y_pred_exp1, Y_test, pairs_test, glove_6b)
delta = exp1_results["top1"] - 30.67

print(f"\nResults (best alpha={best_alpha}):")
print(f"  Top-1:  {exp1_results['top1']}%  (Δ {delta:+.2f}%)")
print(f"  Top-5:  {exp1_results['top5']}%")
print(f"  Top-10: {exp1_results['top10']}%")

# Full alpha sweep
print(f"\nFull alpha sweep:")
for a in alphas:
    r = Ridge(alpha=a)
    r.fit(X_train, Y_train)
    yp = r.predict(X_test)
    res = evaluate(yp, Y_test, pairs_test, glove_6b)
    marker = " ◄ V10" if a == 1.0 else (" ◄ BEST" if a == best_alpha else "")
    print(f"  alpha={a:>8.1f}  →  Top-1: {res['top1']:.2f}%  Top-5: {res['top5']:.2f}%{marker}")

RESULTS["exp1_ridge_cv"] = {
    "best_alpha": best_alpha,
    "cv_folds": 5,
    "top1": exp1_results["top1"],
    "top5": exp1_results["top5"],
    "top10": exp1_results["top10"],
    "delta_v10": round(delta, 2),
}
save_results()

EXPERIMENT 1: Ridge Alpha Cross-Validation



Best alpha: 100.0
  (V10 used alpha=1.0)



Results (best alpha=100.0):
  Top-1:  29.63%  (Δ -1.04%)
  Top-5:  35.45%
  Top-10: 39.18%

Full alpha sweep:


  alpha=     0.0  →  Top-1: 31.34%  Top-5: 38.96%


  alpha=     0.1  →  Top-1: 31.57%  Top-5: 38.81%


...

  alpha=   100.0  →  Top-1: 29.63%  Top-5: 35.45% ◄ BEST


  alpha=   500.0  →  Top-1: 30.37%  Top-5: 34.10%


  alpha=  1000.0  →  Top-1: 30.07%  Top-5: 33.88%


  alpha=  5000.0  →  Top-1: 29.33%  Top-5: 30.22%


  alpha= 10000.0  →  Top-1: 29.33%  Top-5: 29.55%
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 2: Confidence-Weighted Ridge

Hypothesis: anchors with higher confidence should have more influence during training.

Anchor confidence distribution:
- Mean: 0.747, Std: 0.214
- 16% below 0.5, 47% above 0.8
- Range: 0.30 to 1.00

In [7]:
print("=" * 60)
print("EXPERIMENT 2: Confidence-Weighted Ridge")
print("=" * 60)

best_alpha_exp1 = RESULTS["exp1_ridge_cv"]["best_alpha"]
print(f"Using best alpha from Exp 1: {best_alpha_exp1}")

# Show confidence distribution
print(f"\nAnchor confidence distribution (train set):")
print(f"  Mean:  {conf_train.mean():.3f}")
print(f"  Std:   {conf_train.std():.3f}")
print(f"  <0.5:  {(conf_train < 0.5).sum()} ({(conf_train < 0.5).mean()*100:.1f}%)")
print(f"  >0.8:  {(conf_train > 0.8).sum()} ({(conf_train > 0.8).mean()*100:.1f}%)")

# Train with confidence weights
ridge_weighted = Ridge(alpha=best_alpha_exp1)
ridge_weighted.fit(X_train, Y_train, sample_weight=conf_train)
Y_pred_exp2 = ridge_weighted.predict(X_test)

exp2_results = evaluate(Y_pred_exp2, Y_test, pairs_test, glove_6b)
delta = exp2_results["top1"] - 30.67
delta_exp1 = exp2_results["top1"] - RESULTS["exp1_ridge_cv"]["top1"]

print(f"\nResults (weighted, alpha={best_alpha_exp1}):")
print(f"  Top-1:  {exp2_results['top1']}%  (Δ vs V10: {delta:+.2f}%, Δ vs Exp1: {delta_exp1:+.2f}%)")
print(f"  Top-5:  {exp2_results['top5']}%")
print(f"  Top-10: {exp2_results['top10']}%")

RESULTS["exp2_weighted"] = {
    "weighting": True,
    "alpha": best_alpha_exp1,
    "top1": exp2_results["top1"],
    "top5": exp2_results["top5"],
    "top10": exp2_results["top10"],
    "delta_v10": round(delta, 2),
}
save_results()

EXPERIMENT 2: Confidence-Weighted Ridge
Using best alpha from Exp 1: 100.0

Anchor confidence distribution (train set):
  Mean:  0.705
  Std:   0.211
  <0.5:  1041 (19.4%)
  >0.8:  1980 (36.9%)



Results (weighted, alpha=100.0):
  Top-1:  29.03%  (Δ vs V10: -1.64%, Δ vs Exp1: -0.60%)
  Top-5:  35.15%
  Top-10: 38.51%
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 3: CSLS Retrieval

Hypothesis: plain cosine NN suffers from hubness. CSLS penalizes "hub" English words.

**Warning**: V4 showed CSLS caused a 7-point regression at 300d. Running a hub analysis first as a go/no-go check.

In [8]:
print("=" * 60)
print("EXPERIMENT 3: CSLS Retrieval")
print("=" * 60)

# Use the best model so far
best_so_far = "exp1" if RESULTS["exp1_ridge_cv"]["top1"] >= RESULTS.get("exp2_weighted", {}).get("top1", 0) else "exp2"
print(f"Using best model so far: {best_so_far}")

if best_so_far == "exp1":
    Y_pred_best = Y_pred_exp1
else:
    Y_pred_best = Y_pred_exp2

# Hub analysis: what are the most frequent top-1 predictions?
print("\n--- Hub Analysis (Go/No-Go Check) ---")
best_results = evaluate(Y_pred_best, Y_test, pairs_test, glove_6b)
top1_predictions = best_results["predictions"]

pred_counts = Counter(top1_predictions)
top20_hubs = pred_counts.most_common(20)

stopwords = {"the", "of", "and", "to", "a", "in", "is", "it", "for", "on",
             "that", "with", "as", "at", "by", "from", "or", "an", "be",
             "this", "which", "not", "are", "was", "but", "have", "had", "has"}

print(f"\nTop 20 most frequent Top-1 predictions:")
total_preds = len(top1_predictions)
hub_count = 0
for word, count in top20_hubs:
    pct = count / total_preds * 100
    is_sw = word in stopwords
    marker = " ← STOPWORD" if is_sw else ""
    print(f"  {word:>15s}: {count:>4d} ({pct:.1f}%){marker}")
    if is_sw:
        hub_count += count

hub_pct = hub_count / total_preds * 100
print(f"\nStopword hub rate: {hub_count}/{total_preds} ({hub_pct:.1f}%)")

if hub_pct < 10:
    print("\n⚠️  Hub rate < 10% — hubness is NOT the bottleneck.")
    print("    CSLS unlikely to help. Running anyway for completeness.")
    csls_go = False
else:
    print(f"\n✓ Hub rate {hub_pct:.1f}% > 10% — hubness confirmed. Proceeding with CSLS.")
    csls_go = True

EXPERIMENT 3: CSLS Retrieval
Using best model so far: exp1

--- Hub Analysis (Go/No-Go Check) ---



Top 20 most frequent Top-1 predictions:
              the: 1103 (82.3%) ← STOPWORD
               so:   60 (4.5%)
                .:   42 (3.1%)
                2:   40 (3.0%)
                i:   37 (2.8%)
                1:   10 (0.7%)
               10:    8 (0.6%)
              you:    7 (0.5%)
              but:    6 (0.4%) ← STOPWORD
              and:    4 (0.3%) ← STOPWORD
             well:    3 (0.2%)
             both:    3 (0.2%)
               it:    2 (0.1%) ← STOPWORD
              god:    2 (0.1%)
             what:    2 (0.1%)
             time:    1 (0.1%)
              des:    1 (0.1%)
               is:    1 (0.1%) ← STOPWORD
              not:    1 (0.1%) ← STOPWORD
               me:    1 (0.1%)

Stopword hub rate: 1117/1340 (83.4%)

✓ Hub rate 83.4% > 10% — hubness confirmed. Proceeding with CSLS.


In [9]:
print("\nRunning CSLS evaluation (k=10)...")
print("  (This may take a few minutes — computing against full GloVe vocab)")

exp3_results = evaluate_csls(Y_pred_best, Y_test, pairs_test, glove_6b, k=10)
delta = exp3_results["top1"] - 30.67
delta_best = exp3_results["top1"] - best_results["top1"]

print(f"\nResults (CSLS k=10):")
print(f"  Top-1:  {exp3_results['top1']}%  (Δ vs V10: {delta:+.2f}%, Δ vs best NN: {delta_best:+.2f}%)")
print(f"  Top-5:  {exp3_results['top5']}%")
print(f"  Top-10: {exp3_results['top10']}%")

if delta_best < 0:
    print(f"\n⚠️  CSLS hurt by {abs(delta_best):.2f}% — consistent with V4 finding.")
else:
    print(f"\n✓ CSLS improved by {delta_best:.2f}%!")

RESULTS["exp3_csls"] = {
    "csls_k": 10,
    "hubness_check_passed": csls_go,
    "top20_hub_words": [w for w, _ in top20_hubs],
    "hub_pct": round(hub_pct, 2),
    **exp3_results,
    "delta_v10": round(delta, 2),
}
save_results()


Running CSLS evaluation (k=10)...
  (This may take a few minutes — computing against full GloVe vocab)


CSLS eval:   0%|          | 0/1340 [00:00<?, ?it/s]


Results (CSLS k=10):
  Top-1:  29.63%  (Δ vs V10: -1.04%, Δ vs best NN: +0.00%)
  Top-5:  35.45%
  Top-10: 39.18%

✓ CSLS improved by 0.00%!
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 4: Pure 768d Ablation

Hypothesis: the 1536d architecture (768d text + 768d zeros) may only help via implicit regularization. A properly tuned 768d Ridge might match it.

This experiment runs its own RidgeCV — the 1536d optimal alpha is NOT transferable to 768d.

In [10]:
print("=" * 60)
print("EXPERIMENT 4: Pure 768d Ablation")
print("=" * 60)

# Build 768d-only arrays (strip visual padding)
X_train_768 = X_train[:, :768]
X_test_768 = X_test[:, :768]

print(f"Input shape: {X_train_768.shape} (was {X_train.shape})")
print(f"Param count: {768 * 300:,} = {768*300/len(X_train):.1f}:1 ratio (was {1536*300/len(X_train):.1f}:1)")

# Own RidgeCV
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]
ridge_cv_768 = RidgeCV(alphas=alphas, cv=5, scoring="neg_mean_squared_error")
ridge_cv_768.fit(X_train_768, Y_train)

best_alpha_768 = float(ridge_cv_768.alpha_)
print(f"\nBest alpha (768d): {best_alpha_768}")
print(f"Best alpha (1536d): {RESULTS['exp1_ridge_cv']['best_alpha']}")

ridge_768 = Ridge(alpha=best_alpha_768)
ridge_768.fit(X_train_768, Y_train)
Y_pred_768 = ridge_768.predict(X_test_768)

exp4_results = evaluate(Y_pred_768, Y_test, pairs_test, glove_6b)
delta = exp4_results["top1"] - 30.67
delta_1536 = exp4_results["top1"] - RESULTS["exp1_ridge_cv"]["top1"]

print(f"\nResults (768d, alpha={best_alpha_768}):")
print(f"  Top-1:  {exp4_results['top1']}%  (Δ vs V10: {delta:+.2f}%)")
print(f"  Top-5:  {exp4_results['top5']}%")
print(f"  Top-10: {exp4_results['top10']}%")
print(f"\n  vs 1536d (Exp 1): {delta_1536:+.2f}%")

if abs(delta_1536) < 0.5:
    print("  → Negligible difference — zeros were acting as regularization.")
elif delta_1536 > 0:
    print("  → 768d is BETTER — 1536d zeros were adding noise, not helping.")
else:
    print("  → 1536d is better — extra dimensions provide real capacity benefit.")

RESULTS["exp4_768d_ablation"] = {
    "input_dim": 768,
    "best_alpha_768d": best_alpha_768,
    "top1": exp4_results["top1"],
    "top5": exp4_results["top5"],
    "top10": exp4_results["top10"],
    "delta_v10": round(delta, 2),
}
save_results()

EXPERIMENT 4: Pure 768d Ablation
Input shape: (5360, 768) (was (5360, 1536))
Param count: 230,400 = 43.0:1 ratio (was 86.0:1)



Best alpha (768d): 100.0
Best alpha (1536d): 100.0



Results (768d, alpha=100.0):
  Top-1:  29.63%  (Δ vs V10: -1.04%)
  Top-5:  35.6%
  Top-10: 39.18%

  vs 1536d (Exp 1): +0.00%
  → Negligible difference — zeros were acting as regularization.
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 5: GloVe 840B Target Space

Hypothesis: GloVe 840B (2.2M vocab, 840B tokens) will recover anchor coverage lost to GloVe 6B's 400K vocab.

**Prerequisite**: Run `python heiro_v13/scripts/download_glove_840b.py` first.

In [11]:
print("=" * 60)
print("EXPERIMENT 5: GloVe 840B Target Space")
print("=" * 60)

glove_840b_path = REPO_ROOT / "heiro_v13/data/glove.840B.300d.txt"

if not glove_840b_path.exists():
    print(f"⚠️  GloVe 840B not found at {glove_840b_path}")
    print("   Run: python heiro_v13/scripts/download_glove_840b.py")
    print("   Skipping Experiment 5.")
    RESULTS["exp5_glove_840b"] = {"skipped": True, "reason": "file not found"}
    save_results()
else:
    print("Loading GloVe 840B (this may take several minutes)...")
    glove_840b = KeyedVectors.load_word2vec_format(
        str(glove_840b_path), binary=False, no_header=True
    )
    print(f"  ✓ {len(glove_840b)} English words, {glove_840b.vector_size}d")

    # Rebuild anchors against 840B
    X_840, Y_840, conf_840, pairs_840 = [], [], [], []
    for a in anchors:
        egy, eng = a["hieroglyphic"], a["english"].lower()
        if egy in fused_embeddings and eng in glove_840b:
            X_840.append(fused_embeddings[egy])
            Y_840.append(glove_840b[eng])
            conf_840.append(a["confidence"])
            pairs_840.append((egy, eng))

    X_840 = np.array(X_840)
    Y_840 = np.array(Y_840)
    conf_840 = np.array(conf_840)

    coverage_6b = len(X_all)
    coverage_840b = len(X_840)
    print(f"\nAnchor coverage:")
    print(f"  GloVe 6B:   {coverage_6b} / {len(anchors)} ({coverage_6b/len(anchors)*100:.1f}%)")
    print(f"  GloVe 840B: {coverage_840b} / {len(anchors)} ({coverage_840b/len(anchors)*100:.1f}%)")
    print(f"  Delta:      +{coverage_840b - coverage_6b} anchors recovered")

    # Split
    X_tr_840, X_te_840, Y_tr_840, Y_te_840, conf_tr_840, conf_te_840, pairs_tr_840, pairs_te_840 = train_test_split(
        X_840, Y_840, conf_840, pairs_840, test_size=0.2, random_state=42
    )

    # Re-tune alpha against 840B targets
    alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]
    ridge_cv_840 = RidgeCV(alphas=alphas, cv=5, scoring="neg_mean_squared_error")
    ridge_cv_840.fit(X_tr_840, Y_tr_840)
    best_alpha_840 = float(ridge_cv_840.alpha_)
    print(f"\nBest alpha (840B): {best_alpha_840}")

    # Check if weighting helped in earlier experiments
    use_weighting = RESULTS.get("exp2_weighted", {}).get("top1", 0) > RESULTS["exp1_ridge_cv"]["top1"]

    ridge_840 = Ridge(alpha=best_alpha_840)
    if use_weighting:
        print("Using confidence weighting (carried from Exp 2)")
        ridge_840.fit(X_tr_840, Y_tr_840, sample_weight=conf_tr_840)
    else:
        ridge_840.fit(X_tr_840, Y_tr_840)

    Y_pred_840 = ridge_840.predict(X_te_840)
    exp5_results = evaluate(Y_pred_840, Y_te_840, pairs_te_840, glove_840b)
    delta = exp5_results["top1"] - 30.67

    print(f"\nResults (GloVe 840B, alpha={best_alpha_840}):")
    print(f"  Top-1:  {exp5_results['top1']}%  (Δ vs V10: {delta:+.2f}%)")
    print(f"  Top-5:  {exp5_results['top5']}%")
    print(f"  Top-10: {exp5_results['top10']}%")

    RESULTS["exp5_glove_840b"] = {
        "glove_vocab_size": len(glove_840b),
        "anchor_coverage_840b": coverage_840b,
        "anchor_coverage_delta": coverage_840b - coverage_6b,
        "best_alpha_840b": best_alpha_840,
        "weighted": use_weighting,
        "top1": exp5_results["top1"],
        "top5": exp5_results["top5"],
        "top10": exp5_results["top10"],
        "delta_v10": round(delta, 2),
    }
    save_results()

EXPERIMENT 5: GloVe 840B Target Space
⚠️  GloVe 840B not found at /Users/crashy/Development/heiroglyphy/heiro_v13/data/glove.840B.300d.txt
   Run: python heiro_v13/scripts/download_glove_840b.py
   Skipping Experiment 5.
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


## Experiment 6: Alpha=0.1 + Proper CSLS

The alpha sweep in Exp 1 revealed alpha=0.1 → 31.57% (best retrieval accuracy), even though RidgeCV picked alpha=100 by MSE. MSE is anti-correlated with retrieval accuracy for this task.

The Exp 3 CSLS implementation was buggy — it only subtracted `r_T(x)` (constant per query, doesn't change ranking). The proper CSLS formula also subtracts `r_S(y)` — the per-English-word hub penalty — which is what actually demotes hubs like "the" (82.3% of all predictions).

In [12]:
print("=" * 60)
print("V13 EXPERIMENT SUMMARY")
print("=" * 60)

print(f"\n{'Experiment':<30s} {'Top-1':>8s} {'Top-5':>8s} {'Top-10':>8s} {'Δ V10':>8s}")
print("-" * 70)
print(f"{'V10 Baseline':<30s} {'30.67':>8s} {'37.69':>8s} {'41.49':>8s} {'—':>8s}")

all_exps = [
    ("exp1_ridge_cv", "1: Ridge CV (α=100)"),
    ("exp2_weighted", "2: Conf-Weighted"),
    ("exp3_csls", "3: CSLS (buggy)"),
    ("exp4_768d_ablation", "4: 768d Ablation"),
    ("exp5_glove_840b", "5: GloVe 840B"),
    ("exp6a_alpha01", "6a: Alpha=0.1"),
    ("exp6b_csls_proper", "6b: α=0.1 + Proper CSLS"),
]

for key, label in all_exps:
    r = RESULTS.get(key, {})
    if r.get("skipped"):
        print(f"{label:<30s} {'SKIPPED':>8s}")
        continue
    if "top1" not in r:
        continue
    print(f"{label:<30s} {r['top1']:>8.2f} {r['top5']:>8.2f} {r['top10']:>8.2f} {r.get('delta_v10', 0):>+8.2f}")

# Determine best config
best_key = None
best_top1 = 30.67
for key, _ in all_exps:
    r = RESULTS.get(key, {})
    if r.get("top1", 0) > best_top1:
        best_top1 = r["top1"]
        best_key = key

if best_key:
    print(f"\n🏆 Best: {best_key} — {best_top1:.2f}% (Δ {best_top1 - 30.67:+.2f}%)")
    RESULTS["best_config"] = {"description": best_key, "top1": best_top1}
else:
    print(f"\nNo experiment beat V10 baseline.")
    RESULTS["best_config"] = {"description": "v10_baseline", "top1": 30.67}

save_results()
print(f"\nFull results: {RESULTS_PATH}")

V13 EXPERIMENT SUMMARY

Experiment                        Top-1    Top-5   Top-10    Δ V10
----------------------------------------------------------------------
V10 Baseline                      30.67    37.69    41.49        —
1: Ridge CV (α=100)               29.63    35.45    39.18    -1.04
2: Conf-Weighted                  29.03    35.15    38.51    -1.64
3: CSLS (buggy)                   29.63    35.45    39.18    -1.04
4: 768d Ablation                  29.63    35.60    39.18    -1.04
5: GloVe 840B                   SKIPPED

No experiment beat V10 baseline.
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json

Full results: /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


In [13]:
print("=" * 60)
print("EXPERIMENT 6a: Alpha=0.1 (retrieval-optimal)")
print("=" * 60)

ridge_01 = Ridge(alpha=0.1)
ridge_01.fit(X_train, Y_train)
Y_pred_exp6a = ridge_01.predict(X_test)

exp6a_results = evaluate(Y_pred_exp6a, Y_test, pairs_test, glove_6b)
delta = exp6a_results["top1"] - 30.67
print(f"\nResults (alpha=0.1):")
print(f"  Top-1:  {exp6a_results['top1']}%  (Δ vs V10: {delta:+.2f}%)")
print(f"  Top-5:  {exp6a_results['top5']}%")
print(f"  Top-10: {exp6a_results['top10']}%")

# Hub analysis at alpha=0.1
preds_01 = exp6a_results["predictions"]
pred_counts_01 = Counter(preds_01)
top5_hubs_01 = pred_counts_01.most_common(5)
print(f"\nTop 5 predictions at alpha=0.1:")
for w, c in top5_hubs_01:
    print(f"  {w:>15s}: {c:>4d} ({c/len(preds_01)*100:.1f}%)")

RESULTS["exp6a_alpha01"] = {
    "alpha": 0.1,
    "top1": exp6a_results["top1"],
    "top5": exp6a_results["top5"],
    "top10": exp6a_results["top10"],
    "delta_v10": round(delta, 2),
}
save_results()

EXPERIMENT 6a: Alpha=0.1 (retrieval-optimal)



Results (alpha=0.1):
  Top-1:  31.57%  (Δ vs V10: +0.90%)
  Top-5:  38.81%
  Top-10: 42.61%

Top 5 predictions at alpha=0.1:
              the:  979 (73.1%)
                2:   45 (3.4%)
               so:   45 (3.4%)
                i:   37 (2.8%)
              you:   28 (2.1%)
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json


In [14]:
def evaluate_csls_proper(Y_pred, Y_test, pairs_test, english_kv, k=10, topn=10):
    """Proper CSLS with both r_T(x) and r_S(y) terms.
    
    CSLS(x, y) = 2*cos(x,y) - r_T(x) - r_S(y)
    
    r_T(x) = mean sim of source vector x to its k nearest target neighbors
    r_S(y) = mean sim of target word y to its k nearest source neighbors
    
    r_S(y) is precomputed for all English words using the full set of
    predicted vectors. This is what penalizes hub words like "the".
    """
    en_words = english_kv.index_to_key
    en_matrix = english_kv.vectors.astype(np.float32)
    en_norms = np.linalg.norm(en_matrix, axis=1, keepdims=True)
    en_normed = en_matrix / np.maximum(en_norms, 1e-10)

    pred_norms = np.linalg.norm(Y_pred, axis=1, keepdims=True)
    pred_normed = (Y_pred / np.maximum(pred_norms, 1e-10)).astype(np.float32)

    # Precompute r_S(y) for all English words
    # For each English word, find mean similarity to k nearest predicted vectors
    # This is the expensive step — (vocab_size x n_predictions) matrix
    print(f"  Precomputing r_S for {len(en_words)} English words against {len(Y_pred)} predictions...")
    
    # Process in batches to avoid OOM (400K x 1340 is ~2GB float32)
    batch_size = 10000
    r_s = np.zeros(len(en_words), dtype=np.float32)
    
    for start in tqdm(range(0, len(en_words), batch_size), desc="r_S batches"):
        end = min(start + batch_size, len(en_words))
        # (batch, 300) @ (300, n_pred) -> (batch, n_pred)
        sims_batch = en_normed[start:end] @ pred_normed.T
        # Mean of top-k similarities for each English word in this batch
        if sims_batch.shape[1] >= k:
            top_k = np.partition(sims_batch, -k, axis=1)[:, -k:]
            r_s[start:end] = top_k.mean(axis=1)
        else:
            r_s[start:end] = sims_batch.mean(axis=1)

    print(f"  r_S stats: mean={r_s.mean():.4f}, std={r_s.std():.4f}")
    print(f"  Top hub penalties: {sorted(r_s)[-5:]}")

    # Now evaluate with proper CSLS
    correct = {1: 0, 5: 0, 10: 0}

    for i in tqdm(range(len(Y_pred)), desc="CSLS eval"):
        pred_vec = pred_normed[i]
        true_word = pairs_test[i][1]

        # Cosine similarities
        sims = en_normed @ pred_vec

        # r_T(x): mean of k nearest English neighbors for this query
        top_k_sims = np.partition(sims, -k)[-k:]
        r_t = top_k_sims.mean()

        # Proper CSLS: penalizes both query hubness AND target hubness
        csls_scores = 2 * sims - r_t - r_s

        # Top-N by CSLS
        top_indices = np.argpartition(csls_scores, -topn)[-topn:]
        top_indices = top_indices[np.argsort(csls_scores[top_indices])[::-1]]
        words = [en_words[idx] for idx in top_indices]

        if true_word == words[0]:
            correct[1] += 1
        if true_word in words[:5]:
            correct[5] += 1
        if true_word in words[:topn]:
            correct[10] += 1

    n = len(Y_pred)
    return {
        "top1": round(correct[1] / n * 100, 2),
        "top5": round(correct[5] / n * 100, 2),
        "top10": round(correct[10] / n * 100, 2),
    }

print("✓ Proper CSLS function defined")

✓ Proper CSLS function defined


In [15]:
print("=" * 60)
print("EXPERIMENT 6b: Alpha=0.1 + Proper CSLS")
print("=" * 60)

print(f"\nUsing alpha=0.1 model (Top-1: {exp6a_results['top1']}%)")
print("Applying proper CSLS with r_S(y) hub penalty...\n")

# Use ALL predicted vectors (not just test) for r_S computation
# This gives CSLS a better picture of which English words are hubs
all_fused = np.array([fused_embeddings[w] for w in text_embeddings.index_to_key])
Y_pred_all = ridge_01.predict(all_fused)

exp6b_results = evaluate_csls_proper(
    Y_pred_exp6a, Y_test, pairs_test, glove_6b, k=10
)
delta = exp6b_results["top1"] - 30.67
delta_6a = exp6b_results["top1"] - exp6a_results["top1"]

print(f"\nResults (alpha=0.1 + proper CSLS):")
print(f"  Top-1:  {exp6b_results['top1']}%  (Δ vs V10: {delta:+.2f}%, Δ vs 6a: {delta_6a:+.2f}%)")
print(f"  Top-5:  {exp6b_results['top5']}%")
print(f"  Top-10: {exp6b_results['top10']}%")

RESULTS["exp6b_csls_proper"] = {
    "alpha": 0.1,
    "csls_k": 10,
    "csls_version": "proper (r_T + r_S)",
    "top1": exp6b_results["top1"],
    "top5": exp6b_results["top5"],
    "top10": exp6b_results["top10"],
    "delta_v10": round(delta, 2),
}
save_results()

EXPERIMENT 6b: Alpha=0.1 + Proper CSLS

Using alpha=0.1 model (Top-1: 31.57%)
Applying proper CSLS with r_S(y) hub penalty...



  Precomputing r_S for 400000 English words against 1340 predictions...


r_S batches:   0%|          | 0/40 [00:00<?, ?it/s]

  r_S stats: mean=0.0393, std=0.0674
  Top hub penalties: [np.float32(0.8940051), np.float32(0.90565526), np.float32(0.9224695), np.float32(0.9393836), np.float32(0.96500856)]


CSLS eval:   0%|          | 0/1340 [00:00<?, ?it/s]


Results (alpha=0.1 + proper CSLS):
  Top-1:  26.27%  (Δ vs V10: -4.40%, Δ vs 6a: -5.30%)
  Top-5:  33.88%
  Top-10: 37.39%
Results saved to /Users/crashy/Development/heiroglyphy/heiro_v13/results/experiment_results.json
